# Blue Side Advantage in Professional League of Legends

**Name(s):** Nhan Doan, Sean Liu  
**Website Link:** https://nhdoan0412.github.io/LoL-dsc-prj/

This project studies whether playing on Blue side provides a competitive advantage in **2022 professional League of Legends matches**. The analysis focuses on side advantage first, then builds a predictive model that uses early-game information available at 10 minutes to predict whether a team eventually wins or loses.

In [96]:
from dsc80_utils import *

import plotly.io as pio
import plotly.offline as py

pio.renderers.default = "notebook_connected"
py.init_notebook_mode(connected=True)

from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

np.random.seed(42)

assets_dir = Path("assets")
assets_dir.mkdir(exist_ok=True)

## Introduction

League of Legends is a professional esport where two teams compete from opposite sides of the map: Blue side and Red side. These sides are not visually identical in play. They differ in map orientation, draft order, and the way teams approach neutral objectives, so it is reasonable to ask whether side selection is associated with match outcome.

The main question of this project is:

**Does playing on Blue side provide a competitive advantage in 2022 professional League of Legends matches?**

This question matters because professional League of Legends is highly strategic. If Blue side teams win more often than Red side teams, that pattern could affect how analysts think about draft priority, tournament fairness, and early-game strategy.

For this project, I use the 2022 Oracle's Elixir League of Legends match dataset. The most important columns for the analysis are:

- `gameid`: unique match identifier.
- `league`: professional league where the match was played.
- `position`: whether the row describes an individual player or the team as a whole.
- `side`: whether the team played on Blue side or Red side.
- `result`: match outcome, where 1 means win and 0 means loss.
- `goldat10`, `xpat10`, `csat10`: team gold, experience, and creep score at 10 minutes.
- `golddiffat10`, `xpdiffat10`, `csdiffat10`: the team's gold, experience, and creep score difference versus the opponent at 10 minutes.

In [97]:
csv_files = (
    sorted(Path(".").glob("*2022*LoL*esports*match*.csv")) +
    sorted(Path("data").glob("*2022*LoL*esports*match*.csv")) +
    sorted(Path(".").glob("*2022*.csv")) +
    sorted(Path("data").glob("*2022*.csv"))
)

seen = set()
csv_files = [path for path in csv_files if not (path in seen or seen.add(path))]

if len(csv_files) == 0:
    raise FileNotFoundError("No 2022 League of Legends CSV file was found in the project folder or data folder.")

csv_path = csv_files[0]
lol = pd.read_csv(csv_path, low_memory=False)

print("Loaded file:", csv_path)
print("Raw 2022 dataset shape:", lol.shape)
display(lol.head())

Loaded file: 2022_LoL_esports_match_data_from_OraclesElixir.csv
Raw 2022 dataset shape: (150348, 165)


,gameid,datacompleteness,url,league,...,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,1.0,0.0,2.0,0.0
1,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,2.0,1.0,5.0,1.0
2,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,0.0,3.0,4.0,3.0
3,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,2.0,3.0,4.0,0.0
4,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,2.0,0.0,7.0,2.0


## Data Cleaning and Exploratory Data Analysis

The raw dataset contains both player-level rows and team-level rows. Since the research question is about whether an entire team wins from Blue side or Red side, I keep only rows where `position == "team"`. This prevents player-level rows from being mixed with team-level observations.

I also keep only rows with valid `side` and `result` values. Then, I create two useful columns: `result_label`, which changes 0 and 1 into readable labels, and `is_blue`, which records whether the team played on Blue side.

I also converted columns such as `firstblood`, `firstdragon`, `firstherald`, `firstbaron`, `firsttower`, and `playoffs` to pandas nullable boolean type when those columns were present. This addresses the dataset-specific issue that some boolean columns are not originally stored as boolean values.

In [98]:
team = lol[lol["position"] == "team"].copy()

team = team[team["side"].isin(["Blue", "Red"])].copy()
team = team[team["result"].isin([0, 1])].copy()

team["result"] = team["result"].astype(int)

bool_cols = [
    "firstblood",
    "firstdragon",
    "firstherald",
    "firstbaron",
    "firsttower",
    "playoffs"
]

bool_cols = [col for col in bool_cols if col in team.columns]

for col in bool_cols:
    team[col] = team[col].astype("boolean")

team["result_label"] = team["result"].map({1: "Win", 0: "Loss"})
team["is_blue"] = (team["side"] == "Blue").astype(int)

analysis_cols = [
    "gameid", "league", "side", "result",
    "goldat10", "xpat10", "csat10",
    "golddiffat10", "xpdiffat10", "csdiffat10"
]

analysis_cols = [col for col in analysis_cols if col in team.columns]
cleaned = team[analysis_cols].copy()

print("Cleaned team-level dataset shape:", cleaned.shape)
display(cleaned.head())

print(cleaned.head().to_markdown(index=False))

Cleaned team-level dataset shape: (25058, 10)


,gameid,league,side,result,...,csat10,golddiffat10,xpdiffat10,csdiffat10
10,ESPORTSTMNT01_2690210,LCKC,Blue,0,...,322.0,1523.0,137.0,-8.0
11,ESPORTSTMNT01_2690210,LCKC,Red,1,...,330.0,-1523.0,-137.0,8.0
22,ESPORTSTMNT01_2690219,LCKC,Blue,0,...,317.0,-1619.0,-1586.0,-27.0
23,ESPORTSTMNT01_2690219,LCKC,Red,1,...,344.0,1619.0,1586.0,27.0
34,8401-8401_game_1,LPL,Blue,1,...,NaN,NaN,NaN,NaN


| gameid                | league   | side   |   result |   goldat10 |   xpat10 |   csat10 |   golddiffat10 |   xpdiffat10 |   csdiffat10 |
|:----------------------|:---------|:-------|---------:|-----------:|---------:|---------:|---------------:|-------------:|-------------:|
| ESPORTSTMNT01_2690210 | LCKC     | Blue   |        0 |      16218 |    18213 |      322 |           1523 |          137 |           -8 |
| ESPORTSTMNT01_2690210 | LCKC     | Red    |        1 |      14695 |    18076 |      330 |          -1523 |         -137 |            8 |
| ESPORTSTMNT01_2690219 | LCKC     | Blue   |        0 |      14939 |    17462 |      317 |          -1619 |        -1586 |          -27 |
| ESPORTSTMNT01_2690219 | LCKC     | Red    |        1 |      16558 |    19048 |      344 |           1619 |         1586 |           27 |
| 8401-8401_game_1      | LPL      | Blue   |        1 |        nan |      nan |      nan |            nan |          nan |          nan |


In [99]:
relevant_columns = pd.DataFrame({
    "Column": [
        "gameid", "league", "side", "result",
        "goldat10", "xpat10", "csat10",
        "golddiffat10", "xpdiffat10", "csdiffat10"
    ],
    "Description": [
        "Unique identifier for a professional game.",
        "Professional league where the match was played.",
        "Whether the team played on Blue side or Red side.",
        "Match outcome; 1 means win and 0 means loss.",
        "Team gold at 10 minutes.",
        "Team experience at 10 minutes.",
        "Team creep score at 10 minutes.",
        "Team gold difference versus the opponent at 10 minutes.",
        "Team experience difference versus the opponent at 10 minutes.",
        "Team creep score difference versus the opponent at 10 minutes."
    ]
})

relevant_columns = relevant_columns[relevant_columns["Column"].isin(cleaned.columns)]
display(relevant_columns)

print(relevant_columns.to_markdown(index=False))

,Column,Description
0,gameid,Unique identifier for a professional game.
1,league,Professional league where the match was played.
2,side,Whether the team played on Blue side or Red side.
...,...,...
7,golddiffat10,Team gold difference versus the opponent at 10...
8,xpdiffat10,Team experience difference versus the opponent...
9,csdiffat10,Team creep score difference versus the opponen...


| Column       | Description                                                    |
|:-------------|:---------------------------------------------------------------|
| gameid       | Unique identifier for a professional game.                     |
| league       | Professional league where the match was played.                |
| side         | Whether the team played on Blue side or Red side.              |
| result       | Match outcome; 1 means win and 0 means loss.                   |
| goldat10     | Team gold at 10 minutes.                                       |
| xpat10       | Team experience at 10 minutes.                                 |
| csat10       | Team creep score at 10 minutes.                                |
| golddiffat10 | Team gold difference versus the opponent at 10 minutes.        |
| xpdiffat10   | Team experience difference versus the opponent at 10 minutes.  |
| csdiffat10   | Team creep score difference versus the opponent at 10 minutes. |


In [100]:
side_counts = cleaned["side"].value_counts().reset_index()
side_counts.columns = ["side", "count"]

fig_side_counts = px.bar(
    side_counts,
    x="side",
    y="count",
    text="count",
    title="Number of Team-Level Observations by Side",
    labels={"side": "Side", "count": "Number of Team-Level Observations"}
)

fig_side_counts.show()
fig_side_counts.write_html(assets_dir / "side-counts.html", include_plotlyjs="cdn")

In [101]:
fig_goldat10 = px.histogram(
    cleaned.dropna(subset=["goldat10"]),
    x="goldat10",
    nbins=40,
    marginal="box",
    title="Distribution of Team Gold at 10 Minutes",
    labels={"goldat10": "Gold at 10 Minutes"}
)

fig_goldat10.show()
fig_goldat10.write_html(assets_dir / "gold-at-10-distribution.html", include_plotlyjs="cdn")

In [102]:
side_win_rates = (
    cleaned.groupby("side")["result"]
    .mean()
    .reset_index()
    .rename(columns={"result": "win_rate"})
)

fig_side_win = px.bar(
    side_win_rates,
    x="side",
    y="win_rate",
    text=side_win_rates["win_rate"].round(3),
    title="Win Rate by Side",
    labels={"side": "Side", "win_rate": "Win Rate"}
)

fig_side_win.update_yaxes(range=[0, 1])
fig_side_win.show()
fig_side_win.write_html(assets_dir / "win-rate-by-side.html", include_plotlyjs="cdn")

display(side_win_rates)

,side,win_rate
0,Blue,0.52
1,Red,0.48


In [103]:
fig_gold_diff_result = px.box(
    team.dropna(subset=["golddiffat10"]),
    x="result_label",
    y="golddiffat10",
    title="Gold Difference at 10 Minutes by Match Result",
    labels={
        "result_label": "Match Result",
        "golddiffat10": "Gold Difference at 10 Minutes"
    }
)

fig_gold_diff_result.show()
fig_gold_diff_result.write_html(assets_dir / "gold-diff-at-10-by-result.html", include_plotlyjs="cdn")

In [104]:
agg_cols = ["result"]
for col in ["goldat10", "xpat10", "csat10", "golddiffat10", "xpdiffat10", "csdiffat10"]:
    if col in cleaned.columns:
        agg_cols.append(col)

side_summary = (
    cleaned.groupby("side")[agg_cols]
    .mean()
    .round(3)
    .rename(columns={"result": "win_rate"})
)

display(side_summary)
print(side_summary.reset_index().to_markdown(index=False))

,win_rate,goldat10,xpat10,csat10,golddiffat10,xpdiffat10,csdiffat10
side,,,,,,,
Blue,0.53,15733.17,18211.87,314.68,88.16,24.64,0.34
Red,0.47,15645.02,18187.23,314.33,-88.16,-24.64,-0.34


| side   |   win_rate |   goldat10 |   xpat10 |   csat10 |   golddiffat10 |   xpdiffat10 |   csdiffat10 |
|:-------|-----------:|-----------:|---------:|---------:|---------------:|-------------:|-------------:|
| Blue   |      0.525 |    15733.2 |  18211.9 |  314.679 |         88.156 |       24.639 |        0.344 |
| Red    |      0.475 |    15645   |  18187.2 |  314.335 |        -88.156 |      -24.639 |       -0.344 |


## Assessment of Missingness

To decide whether a column is likely NMAR, I need to reason about the data generating process, not just the observed data. Some missingness in this dataset is structural. For example, player-specific fields are missing in team-level rows because those values do not apply to team summaries.

One column that may be NMAR is `url`. This column is missing when there is no official match link or data-provider link recorded for that game. The missingness may depend on the missing value itself: if no official link exists or was recorded, then the value of `url` is missing. To make this missingness MAR instead, I would want an additional column such as `data_provider_present` or `official_match_page_available` that records whether each game had an official tracked source.

For the permutation tests, I analyze missingness in `firstherald`, a gameplay column with non-trivial missingness. I test whether this missingness depends on `league`, and I also test whether it depends on `side`.

In [105]:
missing_rates = (
    team.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_rate")
)

display(missing_rates.head(30))

nontrivial_missing = (
    team.isna()
    .mean()
    .loc[lambda s: (s > 0.01) & (s < 0.95)]
    .sort_values(ascending=False)
)

display(nontrivial_missing.head(30))

,missing_rate
opp_atakhans,1.0
playername,1.0
damageshare,1.0
...,...
opp_killsat25,0.2
deathsat25,0.2
csat25,0.2


void_grubs                 0.94
opp_void_grubs             0.94
monsterkillsenemyjungle    0.85
                           ... 
firstPick                  0.18
turretplates               0.15
opp_turretplates           0.15
Length: 30, dtype: float64

In [106]:
candidate_missing_cols = [
    "firstherald", "firstdragon", "firstbaron",
    "goldat10", "golddiffat10", "xpat10", "xpdiffat10", "csat10", "csdiffat10"
]

missing_col = None
for col in candidate_missing_cols:
    if col in team.columns and 0.01 < team[col].isna().mean() < 0.95:
        missing_col = col
        break

if missing_col is None and len(nontrivial_missing) > 0:
    missing_col = nontrivial_missing.index[0]

if missing_col is None:
    raise ValueError("No column with non-trivial missingness was found.")

print("Column selected for missingness analysis:", missing_col)
print("Missing rate:", team[missing_col].isna().mean())

Column selected for missingness analysis: firstherald
Missing rate: 0.15069039827599967


In [107]:
def tvd_between_missing_groups(data, missing_col, group_col):
    '''
    Calculates total variation distance between the distribution of group_col
    for rows where missing_col is missing and rows where missing_col is not missing.
    '''
    temp = data[[missing_col, group_col]].copy()
    temp["is_missing"] = temp[missing_col].isna()
    temp = temp.dropna(subset=[group_col])

    props = (
        temp.groupby("is_missing")[group_col]
        .value_counts(normalize=True)
        .rename("proportion")
        .reset_index()
    )

    pivoted = props.pivot(index=group_col, columns="is_missing", values="proportion").fillna(0)

    if True not in pivoted.columns:
        pivoted[True] = 0
    if False not in pivoted.columns:
        pivoted[False] = 0

    return 0.5 * np.abs(pivoted[True] - pivoted[False]).sum()


def categorical_missingness_permutation_test(data, missing_col, group_col, repetitions=1000):
    '''
    Tests whether missingness in missing_col depends on a categorical column.
    The test statistic is TVD.
    '''
    temp = data[[missing_col, group_col]].copy()
    temp["is_missing"] = temp[missing_col].isna()
    temp = temp.dropna(subset=[group_col])

    observed = tvd_between_missing_groups(temp, missing_col, group_col)
    simulated = []

    for _ in range(repetitions):
        shuffled = temp.copy()
        shuffled["is_missing"] = np.random.permutation(shuffled["is_missing"].values)
        shuffled["_missing_marker"] = np.where(shuffled["is_missing"], np.nan, 1)
        simulated.append(tvd_between_missing_groups(shuffled, "_missing_marker", group_col))

    simulated = np.array(simulated)
    p_value = np.mean(simulated >= observed)

    return observed, simulated, p_value

In [108]:
observed_league, sim_league, p_league = categorical_missingness_permutation_test(
    team,
    missing_col=missing_col,
    group_col="league",
    repetitions=1000
)

print("Missingness column:", missing_col)
print("Observed TVD for league:", observed_league)
print("P-value for league dependency:", p_league)

fig_missing_league = px.histogram(
    x=sim_league,
    nbins=40,
    title=f"Permutation Test: Does Missingness of {missing_col} Depend on League?",
    labels={"x": "Simulated TVD Under the Null"}
)

fig_missing_league.add_vline(
    x=observed_league,
    line_dash="dash",
    line_width=3,
    annotation_text="Observed TVD"
)

fig_missing_league.show()
fig_missing_league.write_html(assets_dir / "missingness-league-test.html", include_plotlyjs="cdn")

Missingness column: firstherald
Observed TVD for league: 0.9925847457627119
P-value for league dependency: 0.0


In [109]:
observed_side_missing, sim_side_missing, p_side_missing = categorical_missingness_permutation_test(
    team,
    missing_col=missing_col,
    group_col="side",
    repetitions=1000
)

print("Missingness column:", missing_col)
print("Observed TVD for side:", observed_side_missing)
print("P-value for side dependency:", p_side_missing)

fig_missing_side = px.histogram(
    x=sim_side_missing,
    nbins=40,
    title=f"Permutation Test: Does Missingness of {missing_col} Depend on Side?",
    labels={"x": "Simulated TVD Under the Null"}
)

fig_missing_side.add_vline(
    x=observed_side_missing,
    line_dash="dash",
    line_width=3,
    annotation_text="Observed TVD"
)

fig_missing_side.show()
fig_missing_side.write_html(assets_dir / "missingness-side-test.html", include_plotlyjs="cdn")

Missingness column: firstherald
Observed TVD for side: 0.0
P-value for side dependency: 1.0


### Missingness Test Conclusion

The missingness permutation tests suggest that missingness in `firstherald` depends on `league` but not on `side`. This makes sense because different leagues may have different data tracking practices, tournament formats, or coverage quality. In contrast, each match has one Blue team and one Red team, so there is no strong reason for `firstherald` missingness to depend on side.

## Hypothesis Testing

I test whether Blue-side teams win more often than expected under a fair 50/50 side model.

**Null Hypothesis:** Teams playing on Blue side and Red side have equal probabilities of winning. Equivalently, the Blue-side win rate is 50%, and any observed difference is due to random chance.

**Alternative Hypothesis:** Teams playing on Blue side have a higher probability of winning than teams playing on Red side.

**Test Statistic:** Blue-side win rate minus 0.5.

I use a significance level of 0.05. Since the alternative hypothesis says the Blue-side win rate is higher than 50%, this is a right-tailed test.

In [110]:
blue = team[team["side"] == "Blue"].copy()

observed_blue_win_rate = blue["result"].mean()
observed_stat = observed_blue_win_rate - 0.5

print("Observed Blue-side win rate:", observed_blue_win_rate)
print("Observed test statistic:", observed_stat)

Observed Blue-side win rate: 0.5247825045893527
Observed test statistic: 0.024782504589352716


In [111]:
n_blue_games = blue.shape[0]
n_repetitions = 10000

simulated_stats = []

for _ in range(n_repetitions):
    simulated_results = np.random.choice([0, 1], size=n_blue_games, p=[0.5, 0.5])
    simulated_stats.append(simulated_results.mean() - 0.5)

simulated_stats = np.array(simulated_stats)
p_value_blue = np.mean(simulated_stats >= observed_stat)

print("P-value:", p_value_blue)

P-value: 0.0


In [112]:
fig_hypothesis = px.histogram(
    x=simulated_stats,
    nbins=50,
    title="Simulated Blue-Side Win Advantage Under the Null Hypothesis",
    labels={"x": "Simulated Test Statistic: Blue Win Rate - 0.5"}
)

fig_hypothesis.add_vline(
    x=observed_stat,
    line_width=3,
    line_dash="dash",
    annotation_text="Observed Statistic"
)

fig_hypothesis.show()
fig_hypothesis.write_html(assets_dir / "blue-side-hypothesis-test.html", include_plotlyjs="cdn")

alpha = 0.05

if p_value_blue < alpha:
    hypothesis_conclusion = (
        "Since the p-value is below 0.05, I reject the null hypothesis. "
        "There is statistically significant evidence that Blue-side teams win more often "
        "than expected under a 50% null model."
    )
else:
    hypothesis_conclusion = (
        "Since the p-value is at least 0.05, I fail to reject the null hypothesis. "
        "There is not enough evidence to conclude that Blue-side teams win more often "
        "than expected under a 50% null model."
    )

print(hypothesis_conclusion)

Since the p-value is below 0.05, I reject the null hypothesis. There is statistically significant evidence that Blue-side teams win more often than expected under a 50% null model.


### Hypothesis Test Conclusion

Since the p-value is below 0.05, I reject the null hypothesis. There is statistically significant evidence that Blue-side teams win more often than expected under a 50% null model. However, this does not prove that Blue side directly causes teams to win; it only suggests that Blue side is associated with a higher win rate in the 2022 professional League of Legends data.

## Framing a Prediction Problem

My prediction problem is:

**Can we predict whether a team wins a 2022 professional League of Legends match using information available at 10 minutes?**

The response variable is `result`, where 1 means the team won and 0 means the team lost. This is a binary classification problem.

I use accuracy as the main evaluation metric because each match contributes one winning team and one losing team, so the classes are balanced. I also report precision, recall, and F1-score for additional context.

To avoid data leakage, I only use information that would be known around 10 minutes into the game. I do not use final post-game statistics such as final kills, total gold, towers, dragons, barons, or final game length in the main models.

In [113]:
candidate_model_cols = [
    "side",
    "league",
    "goldat10",
    "xpat10",
    "csat10",
    "golddiffat10",
    "xpdiffat10",
    "csdiffat10"
]

model_cols = [col for col in candidate_model_cols if col in team.columns]

model_df = team[["gameid"] + model_cols + ["result"]].copy()
model_df = model_df[model_df["result"].isin([0, 1])].copy()

print("Model columns:", model_cols)
print("Modeling dataframe shape:", model_df.shape)
display(model_df.head())

print(model_df.head().drop(columns=["gameid"]).to_markdown(index=False))

Model columns: ['side', 'league', 'goldat10', 'xpat10', 'csat10', 'golddiffat10', 'xpdiffat10', 'csdiffat10']
Modeling dataframe shape: (25058, 10)


,gameid,side,league,goldat10,...,golddiffat10,xpdiffat10,csdiffat10,result
10,ESPORTSTMNT01_2690210,Blue,LCKC,16218.0,...,1523.0,137.0,-8.0,0
11,ESPORTSTMNT01_2690210,Red,LCKC,14695.0,...,-1523.0,-137.0,8.0,1
22,ESPORTSTMNT01_2690219,Blue,LCKC,14939.0,...,-1619.0,-1586.0,-27.0,0
23,ESPORTSTMNT01_2690219,Red,LCKC,16558.0,...,1619.0,1586.0,27.0,1
34,8401-8401_game_1,Blue,LPL,NaN,...,NaN,NaN,NaN,1


| side   | league   |   goldat10 |   xpat10 |   csat10 |   golddiffat10 |   xpdiffat10 |   csdiffat10 |   result |
|:-------|:---------|-----------:|---------:|---------:|---------------:|-------------:|-------------:|---------:|
| Blue   | LCKC     |      16218 |    18213 |      322 |           1523 |          137 |           -8 |        0 |
| Red    | LCKC     |      14695 |    18076 |      330 |          -1523 |         -137 |            8 |        1 |
| Blue   | LCKC     |      14939 |    17462 |      317 |          -1619 |        -1586 |          -27 |        0 |
| Red    | LCKC     |      16558 |    19048 |      344 |           1619 |         1586 |           27 |        1 |
| Blue   | LPL      |        nan |      nan |      nan |            nan |          nan |          nan |        1 |


In [114]:
unique_games = model_df["gameid"].drop_duplicates()

train_games, test_games = train_test_split(
    unique_games,
    test_size=0.25,
    random_state=42
)

train_mask = model_df["gameid"].isin(train_games)
test_mask = model_df["gameid"].isin(test_games)

X_train = model_df.loc[train_mask, model_cols]
X_test = model_df.loc[test_mask, model_cols]
y_train = model_df.loc[train_mask, "result"].astype(int)
y_test = model_df.loc[test_mask, "result"].astype(int)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training games:", len(train_games))
print("Testing games:", len(test_games))

print("Training class balance:")
print(y_train.value_counts(normalize=True))

print("Testing class balance:")
print(y_test.value_counts(normalize=True))

Training rows: 18792
Testing rows: 6266
Training games: 9396
Testing games: 3133
Training class balance:
result
0    0.5
1    0.5
Name: proportion, dtype: float64
Testing class balance:
result
0    0.5
1    0.5
Name: proportion, dtype: float64


## Baseline Model

My baseline model is a logistic regression classifier using simple early-game information.

The baseline features are:

- Quantitative features: `goldat10`, `xpat10`, and `csat10`
- Nominal feature: `side`
- Ordinal features: none

| Feature | Type | Encoding / Transformation |
|:---|:---|:---|
| `goldat10` | Quantitative | Median imputation, then `StandardScaler` |
| `xpat10` | Quantitative | Median imputation, then `StandardScaler` |
| `csat10` | Quantitative | Median imputation, then `StandardScaler` |
| `side` | Nominal | Most-frequent imputation, then `OneHotEncoder` |

The quantitative features are median-imputed and standardized. The nominal feature `side` is one-hot encoded. All preprocessing and model training are implemented in a single sklearn Pipeline.

In [115]:
baseline_numeric_features = [col for col in ["goldat10", "xpat10", "csat10"] if col in X_train.columns]
baseline_categorical_features = [col for col in ["side"] if col in X_train.columns]

print("Baseline numeric features:", baseline_numeric_features)
print("Baseline categorical features:", baseline_categorical_features)

baseline_numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

baseline_categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

baseline_preprocessor = ColumnTransformer([
    ("num", baseline_numeric_transformer, baseline_numeric_features),
    ("cat", baseline_categorical_transformer, baseline_categorical_features)
])

baseline_model = Pipeline([
    ("preprocessor", baseline_preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_preds)
baseline_precision = precision_score(y_test, baseline_preds)
baseline_recall = recall_score(y_test, baseline_preds)
baseline_f1 = f1_score(y_test, baseline_preds)

print("Baseline accuracy:", baseline_accuracy)
print("Baseline precision:", baseline_precision)
print("Baseline recall:", baseline_recall)
print("Baseline F1:", baseline_f1)
print()
print(classification_report(y_test, baseline_preds))

Baseline numeric features: ['goldat10', 'xpat10', 'csat10']
Baseline categorical features: ['side']
Baseline accuracy: 0.6262368337057134
Baseline precision: 0.6284507957128938
Baseline recall: 0.6176188956271944
Baseline F1: 0.6229877656149388

              precision    recall  f1-score   support

           0       0.62      0.63      0.63      3133
           1       0.63      0.62      0.62      3133

    accuracy                           0.63      6266
   macro avg       0.63      0.63      0.63      6266
weighted avg       0.63      0.63      0.63      6266



In [116]:
baseline_cm = confusion_matrix(y_test, baseline_preds)

fig_baseline_cm = px.imshow(
    baseline_cm,
    text_auto=True,
    title="Baseline Model Confusion Matrix",
    labels=dict(x="Predicted Result", y="Actual Result", color="Count"),
    x=["Loss", "Win"],
    y=["Loss", "Win"]
)

fig_baseline_cm.show()
fig_baseline_cm.write_html(assets_dir / "baseline-confusion-matrix.html", include_plotlyjs="cdn")

### Baseline Model Interpretation

The baseline logistic regression model achieves an accuracy of about 0.62. This is better than random guessing, which would be around 0.50 for this balanced binary classification problem. However, the model is still limited because it only uses raw early-game resource totals and side, without using opponent-relative differences.

## Final Model

For the final model, I add opponent-relative early-game features and engineered features.

The final model uses:

- Quantitative features: `goldat10`, `xpat10`, `csat10`, `golddiffat10`, `xpdiffat10`, and `csdiffat10`
- Nominal features: `side` and `league`
- Engineered features:
  - `overall_resource_at10`: combines raw gold, XP, and CS at 10 minutes.
  - `overall_diff_at10`: combines gold difference, XP difference, and CS difference at 10 minutes.
  - `positive_gold_diff`: whether the team had a positive gold difference at 10 minutes.

These features are useful because League of Legends advantages are often relative. A team can have a normal amount of gold but still be ahead if the opposing team has less gold. The difference columns capture that opponent-relative context.

I use a Random Forest classifier for the final model and tune `n_estimators`, `max_depth`, and `min_samples_leaf` with GridSearchCV. These hyperparameters control the number of trees and how complex each tree can become.

In [117]:
def add_engineered_features(df):
    '''
    Adds early-game resource and advantage features.
    '''
    X_new = df.copy()

    for col in ["goldat10", "xpat10", "csat10", "golddiffat10", "xpdiffat10", "csdiffat10"]:
        if col not in X_new.columns:
            X_new[col] = np.nan

    X_new["overall_resource_at10"] = (
        X_new["goldat10"] / 1000
        + X_new["xpat10"] / 1000
        + X_new["csat10"] / 100
    )

    X_new["overall_diff_at10"] = (
        X_new["golddiffat10"] / 1000
        + X_new["xpdiffat10"] / 1000
        + X_new["csdiffat10"] / 100
    )

    X_new["positive_gold_diff"] = np.where(
        X_new["golddiffat10"].isna(),
        np.nan,
        (X_new["golddiffat10"] > 0).astype(int)
    )

    return X_new

In [118]:
engineered_train = add_engineered_features(X_train)

final_numeric_features = [
    col for col in [
        "goldat10", "xpat10", "csat10",
        "golddiffat10", "xpdiffat10", "csdiffat10",
        "overall_resource_at10", "overall_diff_at10", "positive_gold_diff"
    ]
    if col in engineered_train.columns
]

final_categorical_features = [
    col for col in ["side", "league"]
    if col in X_train.columns
]

print("Final numeric features:", final_numeric_features)
print("Final categorical features:", final_categorical_features)

Final numeric features: ['goldat10', 'xpat10', 'csat10', 'golddiffat10', 'xpdiffat10', 'csdiffat10', 'overall_resource_at10', 'overall_diff_at10', 'positive_gold_diff']
Final categorical features: ['side', 'league']


In [119]:
final_numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

final_categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

final_preprocessor = ColumnTransformer([
    ("num", final_numeric_transformer, final_numeric_features),
    ("cat", final_categorical_transformer, final_categorical_features)
])

final_pipeline = Pipeline([
    ("feature_engineering", FunctionTransformer(add_engineered_features, validate=False)),
    ("preprocessor", final_preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [4, 8, None],
    "classifier__min_samples_leaf": [5, 10]
}

grid_search = GridSearchCV(
    final_pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy:", grid_search.best_score_)

Best parameters: {'classifier__max_depth': 8, 'classifier__min_samples_leaf': 10, 'classifier__n_estimators': 100}
Best cross-validation accuracy: 0.6764060170793218


In [120]:
final_model = grid_search.best_estimator_
final_preds = final_model.predict(X_test)

final_accuracy = accuracy_score(y_test, final_preds)
final_precision = precision_score(y_test, final_preds)
final_recall = recall_score(y_test, final_preds)
final_f1 = f1_score(y_test, final_preds)

print("Final accuracy:", final_accuracy)
print("Final precision:", final_precision)
print("Final recall:", final_recall)
print("Final F1:", final_f1)
print()
print(classification_report(y_test, final_preds))

model_comparison = pd.DataFrame({
    "Model": ["Baseline Logistic Regression", "Final Random Forest"],
    "Accuracy": [baseline_accuracy, final_accuracy],
    "Precision": [baseline_precision, final_precision],
    "Recall": [baseline_recall, final_recall],
    "F1": [baseline_f1, final_f1]
})

display(model_comparison)
print(model_comparison.round(4).to_markdown(index=False))

Final accuracy: 0.6667730609639323
Final precision: 0.6677367576243981
Final recall: 0.6639004149377593
Final F1: 0.6658130601792573

              precision    recall  f1-score   support

           0       0.67      0.67      0.67      3133
           1       0.67      0.66      0.67      3133

    accuracy                           0.67      6266
   macro avg       0.67      0.67      0.67      6266
weighted avg       0.67      0.67      0.67      6266



,Model,Accuracy,Precision,Recall,F1
0,Baseline Logistic Regression,0.63,0.63,0.62,0.62
1,Final Random Forest,0.67,0.67,0.66,0.67


| Model                        |   Accuracy |   Precision |   Recall |     F1 |
|:-----------------------------|-----------:|------------:|---------:|-------:|
| Baseline Logistic Regression |     0.6262 |      0.6285 |   0.6176 | 0.623  |
| Final Random Forest          |     0.6668 |      0.6677 |   0.6639 | 0.6658 |


In [121]:
final_cm = confusion_matrix(y_test, final_preds)

fig_final_cm = px.imshow(
    final_cm,
    text_auto=True,
    title="Final Model Confusion Matrix",
    labels=dict(x="Predicted Result", y="Actual Result", color="Count"),
    x=["Loss", "Win"],
    y=["Loss", "Win"]
)

fig_final_cm.show()
fig_final_cm.write_html(assets_dir / "final-confusion-matrix.html", include_plotlyjs="cdn")

fig_model_comparison = px.bar(
    model_comparison.melt(id_vars="Model", var_name="Metric", value_name="Score"),
    x="Metric",
    y="Score",
    color="Model",
    barmode="group",
    title="Baseline Model vs Final Model Performance",
    labels={"Score": "Score", "Metric": "Evaluation Metric"}
)

fig_model_comparison.update_yaxes(range=[0, 1])
fig_model_comparison.show()
fig_model_comparison.write_html(assets_dir / "model-comparison.html", include_plotlyjs="cdn")

### Final Model Interpretation

The final Random Forest model improves over the baseline model. This improvement is reasonable because the final model uses opponent-relative features such as gold difference, XP difference, and CS difference at 10 minutes. In League of Legends, relative advantage is often more informative than raw resource totals because a team’s strength depends on how far ahead or behind it is compared to its opponent.

## Fairness Analysis

For fairness analysis, I test whether the final model performs differently for Blue-side teams and Red-side teams.

- Group X: Blue-side teams
- Group Y: Red-side teams
- Evaluation metric: accuracy

**Null Hypothesis:** The final model is fair with respect to side. Its accuracy for Blue-side teams and Red-side teams is approximately the same, and any observed difference is due to random chance.

**Alternative Hypothesis:** The final model is unfair with respect to side. Its accuracy for Blue-side teams is different from its accuracy for Red-side teams.

**Test Statistic:** Absolute difference in accuracy between Blue-side teams and Red-side teams.

I use a significance level of 0.05.

In [122]:
fairness_df = X_test.copy()
fairness_df["actual"] = y_test.values
fairness_df["predicted"] = final_preds
fairness_df["correct"] = (fairness_df["actual"] == fairness_df["predicted"]).astype(int)

accuracy_by_side = fairness_df.groupby("side")["correct"].mean()
observed_fairness_stat = abs(accuracy_by_side.loc["Blue"] - accuracy_by_side.loc["Red"])

print("Accuracy by side:")
print(accuracy_by_side)
print("Observed absolute difference in accuracy:", observed_fairness_stat)

Accuracy by side:
side
Blue    0.67
Red     0.67
Name: correct, dtype: float64
Observed absolute difference in accuracy: 0.0019150973507819913


In [123]:
n_repetitions = 1000
simulated_fairness_stats = []

for _ in range(n_repetitions):
    shuffled = fairness_df.copy()
    shuffled["shuffled_side"] = np.random.permutation(shuffled["side"].values)
    shuffled_acc = shuffled.groupby("shuffled_side")["correct"].mean()

    if "Blue" in shuffled_acc.index and "Red" in shuffled_acc.index:
        stat = abs(shuffled_acc.loc["Blue"] - shuffled_acc.loc["Red"])
        simulated_fairness_stats.append(stat)

simulated_fairness_stats = np.array(simulated_fairness_stats)
fairness_p_value = np.mean(simulated_fairness_stats >= observed_fairness_stat)

print("Fairness p-value:", fairness_p_value)

Fairness p-value: 0.887


In [124]:
fig_fairness = px.histogram(
    x=simulated_fairness_stats,
    nbins=40,
    title="Fairness Permutation Test: Accuracy Difference by Side",
    labels={"x": "Simulated Absolute Difference in Accuracy"}
)

fig_fairness.add_vline(
    x=observed_fairness_stat,
    line_dash="dash",
    line_width=3,
    annotation_text="Observed Difference"
)

fig_fairness.show()
fig_fairness.write_html(assets_dir / "fairness-side-test.html", include_plotlyjs="cdn")

if fairness_p_value < 0.05:
    fairness_conclusion = (
        "Since the p-value is below 0.05, I reject the null hypothesis. "
        "There is evidence that the final model performs differently for Blue-side and Red-side teams."
    )
else:
    fairness_conclusion = (
        "Since the p-value is at least 0.05, I fail to reject the null hypothesis. "
        "There is not enough evidence that the final model performs differently for Blue-side and Red-side teams."
    )

print(fairness_conclusion)

Since the p-value is at least 0.05, I fail to reject the null hypothesis. There is not enough evidence that the final model performs differently for Blue-side and Red-side teams.


## ## Summary of Results

The table below summarizes the main values used in the written report.

In [125]:
website_values = pd.DataFrame({
    "Value": [
        "Raw dataset shape",
        "Cleaned team-level dataset shape",
        "Missingness column",
        "Missingness p-value: league",
        "Missingness p-value: side",
        "Observed Blue-side win rate",
        "Observed Blue-side test statistic",
        "Blue-side hypothesis p-value",
        "Baseline accuracy",
        "Final accuracy",
        "Best final-model parameters",
        "Observed fairness statistic",
        "Fairness p-value"
    ],
    "Result": [
        str(lol.shape),
        str(cleaned.shape),
        missing_col,
        p_league,
        p_side_missing,
        observed_blue_win_rate,
        observed_stat,
        p_value_blue,
        baseline_accuracy,
        final_accuracy,
        str(grid_search.best_params_),
        observed_fairness_stat,
        fairness_p_value
    ]
})

display(website_values)
print(website_values.to_markdown(index=False))

,Value,Result
0,Raw dataset shape,"(150348, 165)"
1,Cleaned team-level dataset shape,"(25058, 10)"
2,Missingness column,firstherald
...,...,...
10,Best final-model parameters,"{'classifier__max_depth': 8, 'classifier__min_..."
11,Observed fairness statistic,0.0
12,Fairness p-value,0.89


| Value                             | Result                                                                                            |
|:----------------------------------|:--------------------------------------------------------------------------------------------------|
| Raw dataset shape                 | (150348, 165)                                                                                     |
| Cleaned team-level dataset shape  | (25058, 10)                                                                                       |
| Missingness column                | firstherald                                                                                       |
| Missingness p-value: league       | 0.0                                                                                               |
| Missingness p-value: side         | 1.0                                                                                               |
| Observed Blue-side win rate     